# Sales and Product Data Analysis

## About the Data
This data represents a retail or e-commerce business tracking product catalog details and individual customer purchase orders.

- **product_master.csv**: Contains details about the products sold, including category, brand, unit price, and cost.
- **sales_transactions.csv**: Contains individual sales orders, including customer details, purchase dates, the specific product purchased, and order quantity.

## Import All CSV Files
We will import `pandas` to load all the CSV files found in the project. We also import `matplotlib` and `seaborn` for visualization later.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets using direct pd.read_csv statements
product_df = pd.read_csv('product_master.csv')
sales_df = pd.read_csv('sales_transactions.csv')

## Basic Inspection
Let's check the number of rows in each table, view some sample rows to understand the format, and check for missing values across the columns.

In [ ]:
print(f"Product Master Rows: {len(product_df)}")
print(f"Sales Transactions Rows: {len(sales_df)}")

print("\n--- Product Master Sample ---")
display(product_df.head(3))

print("\n--- Sales Transactions Sample ---")
display(sales_df.head(3))

print("\n--- Missing Values in Sales ---")
print(sales_df.isnull().sum())

## Data Cleaning
We need to clean the data so it's ready for calculations:
1. **Date formatting**: We convert `order_date` to a proper datetime type to make timeline analysis possible.
2. **Numeric conversion**: The `discount_pct` column contains strings with '%' (e.g., '20%'). We'll strip the '%' and turn it into a numeric decimal.

In [ ]:
# Convert order_date to datetime format (day-month-year)
sales_df['order_date'] = pd.to_datetime(sales_df['order_date'], format='%d-%m-%Y')

# Clean discount_pct column (e.g. '20%' -> 0.20)
sales_df['discount_pct'] = sales_df['discount_pct'].str.replace('%', '').astype(float) / 100

# Show the cleaned columns
display(sales_df[['order_date', 'discount_pct']].head(3))

## Feature Engineering
We will create derived columns based on existing data to make our analysis richer.
- Extracting **Year** and **Month** allows us to neatly group sales data periodically.

In [ ]:
# Create new columns for Year and Month
sales_df['order_year'] = sales_df['order_date'].dt.year
sales_df['order_month'] = sales_df['order_date'].dt.month

display(sales_df[['order_date', 'order_year', 'order_month']].head(3))

## Merges and Master Table

### Merging Sales and Products
- **Tables merged**: `sales_df` and `product_df`
- **Key used**: `product_id`
- **Why it's needed**: The sales table tells us *what* was sold (via ID) and *how many* (quantity), but we need the `unit_price` from the product table to calculate the actual money made.
- **New information adding**: Every sales transaction now has product names, categories, and price/cost information attached.

After merging, we will generate a final `revenue` column taking discounts into account.

In [ ]:
# Merge the dataframes
master_df = pd.merge(sales_df, product_df, on='product_id', how='left')

# Calculate the actual revenue generated per order: quantity * unit_price * (1 - discount)
master_df['revenue'] = master_df['quantity'] * master_df['unit_price'] * (1 - master_df['discount_pct'])

display(master_df.head(3))

## KPI Calculation
We compute Key Performance Indicators (KPIs) to snapshot business health. In simple terms, how much money we made and where it came from.

In [ ]:
total_revenue = master_df['revenue'].sum()
total_orders = len(master_df)
avg_order_value = master_df['revenue'].mean()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: ${avg_order_value:,.2f}\n")

top_category = master_df.groupby('category')['revenue'].sum().idxmax()
print(f"Top Selling Category: {top_category}")

top_region = master_df.groupby('region')['revenue'].sum().idxmax()
print(f"Top Performing Region: {top_region}")

## Charts
Visualizing distributions and trends is essential. We will build three simple charts to represent our data structure.

In [ ]:
# Set visual style for consistency
sns.set_theme(style="whitegrid")

# 1. Bar chart for Top Categories
plt.figure(figsize=(8, 5))
category_sales = master_df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=category_sales, x='category', y='revenue', palette='Blues_r')
plt.title('Total Revenue by Product Category')
plt.ylabel('Revenue ($)')
plt.xlabel('Category')
plt.xticks(rotation=45)
plt.show()

**Interpretation**:
- The vertical bar chart clearly ranks product categories by profitability.
- It shows us our primary revenue drivers among the product types.

In [ ]:
# 2. Line chart for Sales Trend over Time
plt.figure(figsize=(10, 5))
# Group by Year-Month
monthly_sales = master_df.groupby(master_df['order_date'].dt.to_period('M'))['revenue'].sum().reset_index()
# Convert period back to timestamp for plotting
monthly_sales['order_date'] = monthly_sales['order_date'].dt.to_timestamp()

sns.lineplot(data=monthly_sales, x='order_date', y='revenue', marker='o', color='purple')
plt.title('Monthly Sales Revenue Trend')
plt.ylabel('Revenue ($)')
plt.xlabel('Date')
plt.xticks(rotation=45)
plt.show()

**Interpretation**:
- The line chart tracks the flow of sales dollars month over month.
- This reveals periods of high performance and low performance for targeted interventions.

In [ ]:
# 3. Histogram of Order Quantities
plt.figure(figsize=(8, 5))
sns.histplot(master_df['quantity'], bins=10, kde=False, color='teal')
plt.title('Distribution of Order Quantities')
plt.ylabel('Number of Orders')
plt.xlabel('Quantity Purchased per Order')
plt.show()

**Interpretation**:
- The histogram demonstrates the frequency order sizes.
- We can see if bulk purchasing or single-item purchasing is more standard amongst our customer base.

## Final Summary
- **About the Data**: We processed and cleaned retail data tying customers mathematically to product costs.
- **Most Important Result**: We built a master dataset by merging products to sales, deriving 'revenue' for each transaction accurately.
- **Key Business Insight**: Pinpointing top-selling regions and categories enables the business to allocate marketing resources strategically.